In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install tqdm torch torchaudio torchvision transformers==4.44.2 accelerate==0.34.2 tqdm pandas numpy multimolecule

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.4/53.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 126.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import torch, sys, platform
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available?", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch: 2.8.0+cpu
CUDA available? False


In [ ]:
# =========================
# Embedding_Data · SECTION 2: RINALMO INIT (+ CSV index→sequence)
# =========================
!pip -q install multimolecule>=0.2.3

import torch, numpy as np
import pandas as pd
from pathlib import Path
import multimolecule
from multimolecule import RnaTokenizer, RiNALMoModel

# ---- Settings ----
DATA_DIR = Path("/content/drive/MyDrive/1149108/RiGAT-CMI/Dataset/CMI-20208")
MODEL_ID = "multimolecule/rinalmo-giga"
POOLING  = "mean"
BATCH_SIZE = 512

def _to_rna(seq: str) -> str:
    s = seq.strip().upper().replace("T", "U")
    allowed = set("ACGUN")
    return "".join(ch if ch in allowed else "N" for ch in s)

def load_order_csv(path: str):
    path = Path(path)
    df = pd.read_csv(path, header=None, sep=r"\s+|\t|,", engine="python")
    if df.shape[1] < 2:
        raise ValueError(f"{path} must have ≥2 column: index and sequence")
    max_idx = int(df.iloc[:, 0].max())
    seqs = [""] * (max_idx + 1)
    for _, row in df.iterrows():
        idx = int(row.iloc[0])
        seq = str(row.iloc[1])
        seqs[idx] = _to_rna(seq)
    empties = [i for i, s in enumerate(seqs) if s == ""]
    if empties:
        raise ValueError(f"miss sequence at index: {empties[:10]} ...")
    return seqs

# tokenizer & model
tokenizer = RnaTokenizer.from_pretrained(MODEL_ID)
model = RiNALMoModel.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)
HIDDEN = int(model.config.hidden_size)
print("RiNALMo hidden_size =", HIDDEN)


#Embedding-data

In [ ]:
# =========================
# Embedding_Data · SECTION 3: EMBEDDING FUNCTIONS
# =========================
import torch, numpy as np
from tqdm import tqdm

def mean_pool(hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    # hidden: (B,L,H); mask: (B,L)
    mask = mask.float().unsqueeze(-1)        # (B,L,1)
    summed = (hidden * mask).sum(dim=1)      # (B,H)
    denom = mask.sum(dim=1).clamp(min=1e-6)  # (B,1)
    return summed / denom

def chunk_iter(seq: str, inner_len: int):
    if len(seq) <= inner_len:
        return [seq]
    step = inner_len  # no overlap;
    return [seq[i:i+inner_len] for i in range(0, len(seq), step)]

@torch.no_grad()
def embed_sequences(sequences, tokenizer, model, batch_size=512, pool="mean", show_progress=True):
    model.eval()
    # Token budget
    MAX_LEN = 1022
    max_len = min(MAX_LEN, getattr(tokenizer, "model_max_length", MAX_LEN))
    inner_len = max_len - 2  # CLS & SEP
    print(f"Using max_len={max_len}, inner_len={inner_len}")

    # chunk
    chunk_buf, owners = [], []
    for idx, seq in enumerate(sequences):
        for ch in chunk_iter(seq, inner_len):
            chunk_buf.append(ch)
            owners.append(idx)

    H = int(model.config.hidden_size)
    acc = [np.zeros(H, dtype=np.float32) for _ in sequences]
    cnt = [0 for _ in sequences]

    rng = range(0, len(chunk_buf), batch_size)
    if show_progress:
        rng = tqdm(rng, desc="Embedding with RiNALMo")

    for start in rng:
        batch = chunk_buf[start:start+batch_size]
        enc = tokenizer(
            batch, return_tensors="pt", padding=True, truncation=True, max_length=max_len
        )
        for k in enc:
            enc[k] = enc[k].to(model.device)

        out = model(**enc)
        hidden = out.last_hidden_state    # (B,L,H)
        attn = enc["attention_mask"].clone()

        # Remove CLS/SEP
        try:
            cls_id = tokenizer.cls_token_id
            sep_id = tokenizer.sep_token_id
            if cls_id is not None:
                attn[:, 0] = 0     # CLS at position 0
            if sep_id is not None:
                ids = enc["input_ids"]
                attn[ids == sep_id] = 0
        except Exception:
            pass

        if pool == "cls" and getattr(tokenizer, "cls_token_id", None) is not None:
            vec = hidden[:, 0, :]                 # (B,H)
        else:
            vec = mean_pool(hidden, attn)         # (B,H)

        vec = vec.detach().cpu().float().numpy()
        subowners = owners[start:start+len(batch)]
        for v, who in zip(vec, subowners):
            acc[who] += v
            cnt[who] += 1

        del out, hidden, enc
        torch.cuda.empty_cache()

    for i in range(len(acc)):
        if cnt[i] == 0:
            raise RuntimeError(f" index {i} no chunk")
        acc[i] /= float(cnt[i])

    return np.stack(acc, axis=0)  # (N,H)


In [ ]:
# =========================
# Embedding_Data · SECTION 4: RUN EMBEDDING & SAVE
# =========================
from pathlib import Path
import json

mi_order_path = DATA_DIR / "miRNA_order.csv"
ci_order_path = DATA_DIR / "circRNA_order.csv"
assert mi_order_path.exists() and ci_order_path.exists(), "missing miRNA_order.csv or circRNA_order.csv"

mi_seqs = load_order_csv(mi_order_path)
ci_seqs = load_order_csv(ci_order_path)
M, N = len(mi_seqs), len(ci_seqs)
print(f"M (miRNA) = {M}, N (circRNA) = {N}")
print("Examples miRNA[0]:", mi_seqs[0][:50], "...")
print("Examples circRNA[0]:", ci_seqs[0][:50], "...")

X_m = embed_sequences(mi_seqs, tokenizer, model, batch_size=BATCH_SIZE, pool=POOLING)
X_c = embed_sequences(ci_seqs, tokenizer, model, batch_size=BATCH_SIZE, pool=POOLING)
print("Embeddings shape:", X_m.shape, X_c.shape)

# save embedding & meta
out_dir = DATA_DIR
np.save(out_dir / "miRNA_rinalmo_giga.npy", X_m.astype(np.float32))
np.save(out_dir / "circRNA_rinalmo_giga.npy", X_c.astype(np.float32))
torch.save(torch.from_numpy(X_m), out_dir / "miRNA_rinalmo_giga.pt")
torch.save(torch.from_numpy(X_c), out_dir / "circRNA_rinalmo_giga.pt")

meta = {
    "model_id": MODEL_ID,
    "pool": POOLING,
    "hidden_size": int(model.config.hidden_size),
    "batch_size": BATCH_SIZE,
    "M": int(X_m.shape[0]),
    "N": int(X_c.shape[0]),
}
with open(out_dir / "rinalmo_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("Saved:")
for fn in ["miRNA_rinalmo_giga.npy","circRNA_rinalmo_giga.npy",
           "miRNA_rinalmo_giga.pt","circRNA_rinalmo_giga.pt","rinalmo_meta.json"]:
    print(" -", out_dir / fn)


In [ ]:
# ===== BUILD EDGES  =====
from pathlib import Path
import pandas as pd, numpy as np, re

DATA_DIR = Path("/content/drive/MyDrive/1149108/RiGAT-CMI/Dataset/CMI-20208")

POS_NAME = DATA_DIR / "20208.csv"         # Change Datasets
MI_NAMESEQ = DATA_DIR / "miRNA.csv"  
CI_NAMESEQ = DATA_DIR / "circRNA.csv"
MI_ORDER   = DATA_DIR / "miRNA_order.csv"     
CI_ORDER   = DATA_DIR / "circRNA_order.csv"

def _to_rna(s: str) -> str:
    s = s.strip().upper().replace("T","U")
    return "".join(ch for ch in s if ch in set("ACGUN"))

# readding pairs (name)
pos = pd.read_csv(POS_NAME, header=None, sep=r"[,\t\s]+", engine="python").iloc[:, :2]
pos.columns = ["mi_name","ci_name"]

# readding name->seq
mis = pd.read_csv(MI_NAMESEQ, header=None, sep=r"[,\t\s]+", engine="python").iloc[:, :2]
cis = pd.read_csv(CI_NAMESEQ, header=None, sep=r"[,\t\s]+", engine="python").iloc[:, :2]
mis.columns = ["name","seq"]; cis.columns = ["name","seq"]
mis["seq_n"] = mis["seq"].astype(str).map(_to_rna)
cis["seq_n"] = cis["seq"].astype(str).map(_to_rna)

mi_name2seq = dict(zip(mis["name"].astype(str), mis["seq_n"]))
ci_name2seq = dict(zip(cis["name"].astype(str), cis["seq_n"]))

# readding order index->seq, create seq->index
mio = pd.read_csv(MI_ORDER, header=None, sep=r"[,\t\s]+", engine="python").iloc[:, :2]
cio = pd.read_csv(CI_ORDER, header=None, sep=r"[,\t\s]+", engine="python").iloc[:, :2]
mio.columns = ["index","seq"]; cio.columns = ["index","seq"]
mio["index"] = mio["index"].astype(int); cio["index"] = cio["index"].astype(int)
mio["seq_n"] = mio["seq"].astype(str).map(_to_rna)
cio["seq_n"] = cio["seq"].astype(str).map(_to_rna)

mi_seq2idx = dict(zip(mio["seq_n"], mio["index"]))
ci_seq2idx = dict(zip(cio["seq_n"], cio["index"]))

# map name -> seq -> index
mi_idx, ci_idx, missing = [], [], []
for mi_name, ci_name in zip(pos["mi_name"].astype(str), pos["ci_name"].astype(str)):
    mi_s = mi_name2seq.get(mi_name); ci_s = ci_name2seq.get(ci_name)
    if (mi_s is None) or (ci_s is None):
        missing.append((mi_name, ci_name, "name_not_in_name_seq"))
        continue
    mi_i = mi_seq2idx.get(mi_s); ci_i = ci_seq2idx.get(ci_s)
    if (mi_i is None) or (ci_i is None):
        missing.append((mi_name, ci_name, "seq_not_in_order"))
        continue
    mi_idx.append(mi_i); ci_idx.append(ci_i)

if missing:
    print(f"Cannot map {len(missing)} . Ex:", missing[:5])

mi_idx = np.array(mi_idx, dtype=np.int64)
ci_idx = np.array(ci_idx, dtype=np.int64)

n_mi = int(np.load(DATA_DIR/"miRNA_rinalmo_giga.npy").shape[0])
n_ci = int(np.load(DATA_DIR/"circRNA_rinalmo_giga.npy").shape[0])
assert mi_idx.min() >= 0 and mi_idx.max() < n_mi
assert ci_idx.min() >= 0 and ci_idx.max() < n_ci

np.save(DATA_DIR/"edges_mi_idx.npy", mi_idx)
np.save(DATA_DIR/"edges_ci_idx.npy", ci_idx)
print("Saved edges_mi_idx.npy & edges_ci_idx.npy at:", DATA_DIR)
print("Total positives mapped:", len(mi_idx))


In [ ]:
# 5 fold + independent
FOLD_DIR  = DATA_DIR / "5fold_CV"
INDEP_DIR = DATA_DIR / "independent_test"
FOLD_DIR.mkdir(parents=True, exist_ok=True)
INDEP_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR  =", DATA_DIR)
print("FOLD_DIR  =", FOLD_DIR)
print("INDEP_DIR =", INDEP_DIR)


In [ ]:
# =========================
# Embedding_Data · SECTION 5: SPLITS (80/20 + 5-FOLD)
# =========================
from sklearn.model_selection import KFold

# Readding positives  (index mapped)
u = np.load(DATA_DIR / "edges_mi_idx.npy").astype(np.int64).ravel()
v = np.load(DATA_DIR / "edges_ci_idx.npy").astype(np.int64).ravel()
assert len(u) == len(v)
pos_all = np.column_stack([u, v])

n_mi, n_ci = X_m.shape[0], X_c.shape[0]

def generate_negatives(n_mi, n_ci, pos_edges, num_neg, seed=42):
    rng = np.random.default_rng(seed)
    pos_set = set(map(tuple, pos_edges.tolist()))
    neg = []
    tries = 0
    while len(neg) < num_neg:
        i = int(rng.integers(0, n_mi))
        j = int(rng.integers(0, n_ci))
        if (i, j) not in pos_set:
            neg.append((i, j))
        tries += 1
        if tries > num_neg * 50:  # fallback
            for ii in range(n_mi):
                if len(neg) >= num_neg: break
                jj = int(rng.integers(0, n_ci))
                if (ii, jj) not in pos_set: neg.append((ii, jj))
            break
    return np.asarray(neg, dtype=np.int64)

# 80/20 (reproducible)
rng = np.random.default_rng(42)
perm = rng.permutation(len(pos_all))
cut  = int(0.80 * len(pos_all))
pos_train80 = pos_all[perm[:cut]]
pos_indep20 = pos_all[perm[cut:]]

neg_train80 = generate_negatives(n_mi, n_ci, pos_train80, len(pos_train80), seed=43)
neg_indep20 = generate_negatives(n_mi, n_ci, pos_indep20, len(pos_indep20), seed=44)

# independent test 
pd.DataFrame(pos_indep20).to_csv(INDEP_DIR / "pos.csv", sep="\t", header=False, index=False)
pd.DataFrame(neg_indep20).to_csv(INDEP_DIR / "neg.csv", sep="\t", header=False, index=False)

print("Independent saved to:", INDEP_DIR / "pos.csv", "|", INDEP_DIR / "neg.csv")

# 5-fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_id, (tr_idx, te_idx) in enumerate(kf.split(pos_train80)):
    fold_dir = FOLD_DIR / f"fold{fold_id}"
    fold_dir.mkdir(exist_ok=True, parents=True)

    pos_tr = pos_train80[tr_idx]
    pos_te = pos_train80[te_idx]
    neg_tr = generate_negatives(n_mi, n_ci, pos_tr, len(pos_tr), seed=100+fold_id)
    neg_te = generate_negatives(n_mi, n_ci, pos_te, len(pos_te), seed=200+fold_id)

    # Saved CSV 
    pd.DataFrame(pos_tr).to_csv(fold_dir / "pos_train.csv", sep="\t", header=False, index=False)
    pd.DataFrame(pos_te).to_csv(fold_dir / "pos_test.csv",  sep="\t", header=False, index=False)
    pd.DataFrame(neg_tr).to_csv(fold_dir / "neg_train.csv", sep="\t", header=False, index=False)
    pd.DataFrame(neg_te).to_csv(fold_dir / "neg_test.csv",  sep="\t", header=False, index=False)

    print(f"Saved {fold_dir}")
